# Solar Filament Segmentation Challenge 2026
Run the full pipeline on the official test JPEGs. The default path loads the final submitted checkpoints already in this repository and runs on CPU. `FILSEG_FINAL` (or `FINAL` in the setup cell) also replays the documented variants: `three-verifiers` (three verifiers), `single-unet` (one U-Net) and the ablation `public-ensemble` (adds ten public U-Nets). The optional training section uses the original recipes and writes separate checkpoints.

The image model normalizes grayscale H-alpha intensities inside the solar disk. Two team U-Nets predict probabilities under eight rotations and reflections. A threshold tree proposes connected masks. LightGBM and a ResNet-18 verifier estimate the benefit of each proposal, then dynamic programming picks pixel-disjoint instances.

## 1. Setup
Install the dependencies in the [README](README.md). Point `IMAGES` at a directory of official test JPEGs. Use a one-image folder for a quick CPU check. Set `DEVICE = "cuda"` for a full replay on a supported GPU.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd()
IMAGES = Path(os.environ.get("FILSEG_IMAGES", "test_images"))
OUTPUT = Path(os.environ.get("FILSEG_OUT", "data/pipeline/submission.csv"))
DEVICE = os.environ.get("FILSEG_DEVICE", "cpu")
CPU_THREADS = 4
FINAL = os.environ.get("FILSEG_FINAL", "submitted").lower()
RUN_TRAINING = False
TRAIN_INPUT = Path("data/repro/input")
TRAIN_OUTPUT = Path("data/pipeline-training")

if FINAL not in {"public-ensemble", "three-verifiers", "submitted", "single-unet"}:
    raise ValueError("Use submitted, three-verifiers, single-unet or public-ensemble.")
if not (ROOT / "submit.py").is_file():
    raise FileNotFoundError("Open this notebook from the repository root.")
if not IMAGES.is_dir():
    raise FileNotFoundError(f"Set IMAGES to the official test JPEG directory. Missing: {IMAGES}")

def run(*args, env=None):
    """Use the same Python environment as the notebook."""
    subprocess.run([sys.executable, *map(str, args)], cwd=ROOT, env=env, check=True)

## 2. Optional training
Leave `RUN_TRAINING = False` for the submitted checkpoints. To train from scratch on Linux, prepare the official training JSON, training and test JPEGs, fold assignments and GONG neighbour files under `TRAIN_INPUT`. The [reproduction guide](docs/REPRODUCE.md) documents these inputs and the ImageNet initialization. Training requires two CUDA devices. It runs k06/k09/k11 held-out controls, the two final k07 U-Nets, the k17 verifier and the LightGBM fit.

The public-ensemble ablation also uses [Afshar's public ensemble](https://www.kaggle.com/datasets/dariushafshar/solar-filament-unet-weights), whose weights are in `weights/public/`. The optional trainer here produces the team-only recipes (submitted uses its seed-0 verifier).

In [ ]:
import hashlib

if RUN_TRAINING and FINAL == "public-ensemble":
    raise ValueError("The public-ensemble recipe uses released checkpoints only; select a team-only recipe for retraining.")
if RUN_TRAINING:
    run("-m", "training.retrain_team_models", "--input", TRAIN_INPUT, "--out", TRAIN_OUTPUT)
    weights = TRAIN_OUTPUT / "weights"
else:
    weights = ROOT / "weights"
    manifest = "release_weights_public_ensemble.sha256" if FINAL == "public-ensemble" else "release_weights.sha256"
    for row in (ROOT / manifest).read_text(encoding="ascii").splitlines():
        expected, name = row.split()
        checkpoint = weights / name
        if not checkpoint.is_file():
            raise FileNotFoundError(f"Missing bundled {FINAL} file: {checkpoint}. Check the Git checkout.")
        with checkpoint.open("rb") as stream:
            actual = hashlib.file_digest(stream, "sha256").hexdigest()
        if actual != expected:
            raise ValueError(f"Checkpoint checksum mismatch: {name}")
    print("Release checkpoint checksums verified.")

if FINAL == "public-ensemble":
    selector = weights / "public_ensemble" / "lgb_pubown_nometa.txt"
    verifier_files = [weights / "public_ensemble" / f"verifier_resnet18_s{seed}.pt" for seed in range(3)]
    receipt = weights / "public" / "solar_unet_receipt.json"
    public_args = ["--public-receipt", receipt]
else:
    selector = weights / "own_ver3_a05_l025_nometa_lgb.txt"
    # submitted: three-verifiers with the seed-0 verifier only. single-unet: submitted with the seed-0 U-Net only.
    seeds = range(1) if FINAL in {"submitted", "single-unet"} else range(3)
    verifier_files = [weights / f"verifier_resnet18_s{seed}.pt" for seed in seeds]
    public_args = []
unet_files = [weights / f"effv2s_s{seed}_fold-1.pt" for seed in range(1 if FINAL == "single-unet" else 2)]

## 3. Inference
The notebook calls [submit.py](submit.py) with the released penalty and checkpoints. The pipeline uses the JPEG pixels for prediction and the image stem for CSV IDs.

In [ ]:
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
runtime = dict(os.environ, OMP_NUM_THREADS=str(CPU_THREADS), MKL_NUM_THREADS=str(CPU_THREADS))
run(
    "submit.py", "--images", IMAGES, "--out", OUTPUT,
    "--device", DEVICE, "--tta-batch", 1 if DEVICE == "cpu" else 4,
    "--weights", *unet_files,
    "--final", FINAL, "--selector", selector,
    "--verifier-weights", *verifier_files, *public_args,
    env=runtime,
)

## 4. Validate the CSV
Check unique IDs, mask sizes and pixel disjointness. A valid image can have zero predicted filaments, in which case it has no CSV row.

In [ ]:
run("tools/check_submission.py", OUTPUT, "--images", IMAGES, "--allow-empty-images")

## 5. Inspect predictions
Cyan contours show predicted filament boundaries on up to two JPEGs.

In [ ]:
import csv
from collections import defaultdict

import numpy as np
from IPython.display import display
from PIL import Image
from scipy.ndimage import binary_erosion

from filseg.rle import counts_to_mask

by_image = defaultdict(list)
with OUTPUT.open(encoding="utf-8", newline="") as stream:
    for row in csv.DictReader(stream):
        stem = row["filament_id"].rsplit("_", 1)[0]
        by_image[stem].append(row["segmentation_rle"])

images = sorted(p for p in IMAGES.iterdir() if p.is_file() and p.suffix.lower() in (".jpg", ".jpeg"))
examples = [p for p in images if by_image[p.stem]][:2] or images[:2]
for path in examples:
    with Image.open(path) as jpeg:
        preview = np.array(jpeg.convert("RGB"))
    for counts in by_image[path.stem]:
        mask = counts_to_mask(counts).astype(bool)
        preview[mask & ~binary_erosion(mask, iterations=3)] = (0, 240, 255)
    print(f"{path.stem}  |  {len(by_image[path.stem])} predicted filaments")
    display(Image.fromarray(preview).resize((640, 640), Image.Resampling.LANCZOS))